# ADAS Vision - Phase 7: Bigger Model + Augmentation

Upgrades Phase 6c in two ways, both aimed at the weak frames (bright haze, night, hilly under-segmentation):

1. **Bigger model** - DeepLabV3 MobileNetV3-Large (11M params, ASPP multi-scale head) instead of LRASPP (3M). Still MobileNet backbone, so it stays CPU-deployable (~2.6 fps @768x432 on a laptop).
2. **Rich augmentation** - synthetic night, fog/haze, motion blur, and cast shadows, plus a Dice loss term (better IoU).

**Reuses Phase 6c's work:** same IDD data and the same rasterized-mask cache on Drive, so setup is fast if you already ran 6c.

**Survives free-tier disconnects:** per-epoch checkpoint + auto-resume. If the session dies, just Runtime -> Run all again.

## Setup
1. Runtime -> Change runtime type -> **T4 GPU**
2. The 18 GB IDD archive should already be in `MyDrive/adas/` (from Phase 6c). If not, see phase6c.
3. Run all.

> Advisory / research only - never connect any of this to a vehicle's controls.

In [ ]:
# 1) GPU + Drive
!nvidia-smi -L
import torch, os, glob, tarfile, json, time, random
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

from google.colab import drive
drive.mount("/content/drive")
ADAS = "/content/drive/MyDrive/adas"
CKPT = os.path.join(ADAS, "phase7_ckpt.pth")               # resumable checkpoint (Phase 7)
BEST = os.path.join(ADAS, "drivable_idd_deeplab_best.pth")  # best weights (Phase 7)
MASK_CACHE = os.path.join(ADAS, "idd_masks_cache.tar")      # SHARED with Phase 6c - reused

In [ ]:
# 2) Extract the dataset to fast local disk (same as Phase 6c)
DATA = "/content/idd"
if not glob.glob(os.path.join(DATA, "**", "leftImg8bit"), recursive=True):
    os.makedirs(DATA, exist_ok=True)
    archives = [p for p in glob.glob(os.path.join(ADAS, "*.tar.gz")) + glob.glob(os.path.join(ADAS, "*.tgz"))
                if "lite" not in os.path.basename(p).lower()]
    assert archives, "No full-IDD archive in MyDrive/adas/ - see Phase 6c"
    for arc in archives:
        print("extracting", os.path.basename(arc), "...")
        t0 = time.time()
        with tarfile.open(arc) as t:
            t.extractall(DATA)
        print(f"  done in {time.time()-t0:.0f}s")

ROOT = os.path.dirname(glob.glob(os.path.join(DATA, "**", "leftImg8bit"), recursive=True)[0])
print("dataset root:", ROOT)

In [ ]:
# 3) Rasterize polygon labels -> binary masks (RESTORES the Phase 6c cache if present)
import numpy as np
from PIL import Image, ImageDraw
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor

MASKS = "/content/masks"
DRIVABLE_LABELS = {"road", "parking", "drivable fallback"}
MW, MH = 960, 540

def rasterize_one(args):
    jp, outp = args
    try:
        with open(jp) as f:
            d = json.load(f)
        w, h = d.get("imgWidth", 1920), d.get("imgHeight", 1080)
        m = Image.new("L", (MW, MH), 0); draw = ImageDraw.Draw(m)
        sx, sy = MW / w, MH / h
        for obj in d.get("objects", []):
            if obj.get("deleted"):
                continue
            if str(obj.get("label", "")).lower() in DRIVABLE_LABELS:
                poly = [(x * sx, y * sy) for x, y in obj.get("polygon", [])]
                if len(poly) >= 3:
                    draw.polygon(poly, fill=255)
        os.makedirs(os.path.dirname(outp), exist_ok=True)
        m.save(outp); return True
    except Exception as e:
        return f"{jp}: {e}"

if os.path.exists(MASK_CACHE) and not os.path.isdir(MASKS):
    print("restoring mask cache from Drive (shared with Phase 6c) ...")
    with tarfile.open(MASK_CACHE) as t:
        t.extractall("/content")

if not os.path.isdir(MASKS):
    jobs = []
    for split in ("train", "val"):
        for jp in glob.glob(os.path.join(ROOT, "gtFine", split, "*", "*_polygons.json")):
            rel = os.path.relpath(jp, os.path.join(ROOT, "gtFine"))
            outp = os.path.join(MASKS, rel.replace("_gtFine_polygons.json", "_drivable.png")
                                          .replace("_polygons.json", "_drivable.png"))
            jobs.append((jp, outp))
    print(f"rasterizing {len(jobs)} label files ...")
    errs = []
    with ProcessPoolExecutor(max_workers=4) as ex:
        for r in tqdm(ex.map(rasterize_one, jobs, chunksize=64), total=len(jobs)):
            if r is not True:
                errs.append(r)
    print(f"done, {len(errs)} errors")
    with tarfile.open(MASK_CACHE, "w") as t:
        t.add(MASKS, arcname="masks")
print("masks ready")

In [ ]:
# 4) Pair images with masks + AUGMENTED dataset (night / fog / blur / shadow)
import cv2
from torch.utils.data import Dataset, DataLoader

def pairs(split):
    out = []
    for ip in sorted(glob.glob(os.path.join(ROOT, "leftImg8bit", split, "*", "*.*"))):
        base = os.path.splitext(os.path.basename(ip))[0].replace("_leftImg8bit", "").replace("_image", "")
        seq = os.path.basename(os.path.dirname(ip))
        cands = glob.glob(os.path.join(MASKS, split, seq, base + "*_drivable.png"))
        if cands:
            out.append((ip, cands[0]))
    return out

train_pairs, val_pairs = pairs("train"), pairs("val")
print(f"train: {len(train_pairs)} | val: {len(val_pairs)}")
assert len(train_pairs) > 1000, "pairing looks wrong"

IN_W, IN_H = 768, 432
MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD  = np.array([0.229, 0.224, 0.225], np.float32)

# ---- Augmentations (operate on float32 RGB 0-255) ------------------------
def aug_night(img):
    img = np.clip(img * random.uniform(0.35, 0.60), 0, 255)          # darken
    mu = img.mean()
    img = np.clip((img - mu) * random.uniform(0.7, 0.9) + mu, 0, 255)  # lower contrast
    img[..., 2] = np.clip(img[..., 2] * 1.05, 0, 255)                # slight cool cast
    return img

def aug_fog(img):
    h = img.shape[0]
    grad = np.linspace(1.0, 0.3, h).reshape(h, 1, 1)                 # denser toward horizon
    a = random.uniform(0.2, 0.5) * grad
    haze = np.full_like(img, random.uniform(180, 220))
    return np.clip(img * (1 - a) + haze * a, 0, 255)

def aug_motion_blur(img):
    k = random.choice([5, 7, 9]); kern = np.zeros((k, k), np.float32)
    if random.random() < 0.5:
        kern[k // 2, :] = 1.0 / k
    else:
        kern[:, k // 2] = 1.0 / k
    return cv2.filter2D(img, -1, kern)

def aug_shadow(img):
    h, w = img.shape[:2]
    x1, x2 = sorted(random.sample(range(w), 2))
    poly = np.array([[x1, 0], [x2, 0],
                     [min(w, x2 + random.randint(-w // 4, w // 4)), h],
                     [max(0, x1 + random.randint(-w // 4, w // 4)), h]], np.int32)
    m = np.zeros((h, w), np.uint8); cv2.fillPoly(m, [poly], 1)
    img[m == 1] = np.clip(img[m == 1] * random.uniform(0.4, 0.7), 0, 255)
    return img

class IDDAug(Dataset):
    def __init__(self, pair_list, train=True):
        self.pairs, self.train = pair_list, train
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, i):
        ip, mp = self.pairs[i]
        img = cv2.cvtColor(cv2.imread(ip), cv2.COLOR_BGR2RGB)
        lbl = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (IN_W, IN_H)).astype(np.float32)
        lbl = cv2.resize(lbl, (IN_W, IN_H), interpolation=cv2.INTER_NEAREST)
        if self.train:
            if random.random() < 0.5:
                img, lbl = img[:, ::-1].copy(), lbl[:, ::-1].copy()
            if random.random() < 0.25: img = aug_night(img)
            if random.random() < 0.20: img = aug_fog(img)
            if random.random() < 0.20: img = aug_motion_blur(img)
            if random.random() < 0.20: img = aug_shadow(img)
            if random.random() < 0.30:
                img = np.clip(img * random.uniform(0.7, 1.3), 0, 255)
        x = (img / 255.0 - MEAN) / STD
        x = torch.from_numpy(np.ascontiguousarray(x.transpose(2, 0, 1))).float()
        y = torch.from_numpy(np.ascontiguousarray((lbl > 127).astype(np.int64)))
        return x, y

train_dl = DataLoader(IDDAug(train_pairs, True),  batch_size=8, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(IDDAug(val_pairs,  False), batch_size=8, shuffle=False, num_workers=2, pin_memory=True)
print("augmented dataloaders ready")

In [ ]:
# 5) Preview the augmentations (sanity-check they look realistic, not broken)
import matplotlib.pyplot as plt
random.seed(0)
ip, mp = train_pairs[len(train_pairs)//2]
base = cv2.cvtColor(cv2.resize(cv2.imread(ip), (IN_W, IN_H)), cv2.COLOR_BGR2RGB).astype(np.float32)
demos = {"original": base.copy(), "night": aug_night(base.copy()), "fog": aug_fog(base.copy()),
         "motion blur": aug_motion_blur(base.copy()), "shadow": aug_shadow(base.copy())}
fig, axes = plt.subplots(1, len(demos), figsize=(22, 4))
for ax, (name, im) in zip(axes, demos.items()):
    ax.imshow(im.astype(np.uint8)); ax.set_title(name); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# 6) DeepLabV3 MobileNetV3 + CE+Dice(+aux) loss + RESUMABLE training
import torchvision
from torch import nn

EPOCHS = 28
model = torchvision.models.segmentation.deeplabv3_mobilenet_v3_large(
    weights=None, weights_backbone="DEFAULT", num_classes=2, aux_loss=True).to(DEVICE)

opt = torch.optim.AdamW([
    {"params": model.backbone.parameters(),   "lr": 1e-4},
    {"params": model.classifier.parameters(), "lr": 1e-3},
    {"params": model.aux_classifier.parameters(), "lr": 1e-3},
], weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler()
ce = nn.CrossEntropyLoss()

def dice_loss(logits, target, eps=1.0):
    prob = torch.softmax(logits, 1)[:, 1]
    t = (target == 1).float()
    inter = (prob * t).sum((1, 2)); union = prob.sum((1, 2)) + t.sum((1, 2))
    return (1 - (2 * inter + eps) / (union + eps)).mean()

def seg_loss(out, y):
    l = ce(out["out"], y) + dice_loss(out["out"], y)
    if "aux" in out:
        l = l + 0.4 * ce(out["aux"], y)
    return l

start_ep, best_iou = 0, 0.0
if os.path.exists(CKPT):                       # auto-resume
    ck = torch.load(CKPT, map_location=DEVICE)
    model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
    sched.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
    start_ep, best_iou = ck["epoch"] + 1, ck["best_iou"]
    print(f"resumed from epoch {start_ep} (best IoU {best_iou:.3f})")

def val_iou():
    model.eval(); inter = union = 0
    with torch.no_grad():
        for x, y in val_dl:
            p = model(x.to(DEVICE, non_blocking=True))["out"].argmax(1).cpu()
            inter += ((p == 1) & (y == 1)).sum().item()
            union += ((p == 1) | (y == 1)).sum().item()
    return inter / max(union, 1)

for ep in range(start_ep, EPOCHS):
    model.train(); running = 0.0
    for x, y in tqdm(train_dl, desc=f"epoch {ep+1}/{EPOCHS}"):
        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast():
            loss = seg_loss(model(x.to(DEVICE, non_blocking=True)), y.to(DEVICE, non_blocking=True))
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        running += loss.item() * x.size(0)
    sched.step()
    iou = val_iou()
    print(f"epoch {ep+1}: loss {running/len(train_pairs):.4f} | val drivable-IoU {iou:.3f}")
    if iou > best_iou:
        best_iou = iou; torch.save(model.state_dict(), BEST)
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                "scaler": scaler.state_dict(), "epoch": ep, "best_iou": best_iou}, CKPT)
print(f"training complete - best val IoU {best_iou:.3f} (weights: adas/drivable_idd_deeplab_best.pth)")

In [ ]:
# 7) THE TEST - same six dashcam frames (compare against Phase 6c / 15000 decisive)
VIDEO_PATH = os.path.join(ADAS, "dashcam.mp4")
TEST_FRAMES = [2000, 5000, 9000, 12000, 15000, 17000]
model.load_state_dict(torch.load(BEST, map_location=DEVICE)); model.eval()

def infer_drivable(frame_bgr, in_w=768, in_h=432):
    h, w = frame_bgr.shape[:2]
    img = cv2.cvtColor(cv2.resize(frame_bgr, (in_w, in_h)), cv2.COLOR_BGR2RGB)
    x = (img.astype(np.float32) / 255.0 - MEAN) / STD
    x = torch.from_numpy(x.transpose(2, 0, 1)).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        p = model(x)["out"].argmax(1).squeeze().cpu().numpy().astype(np.uint8)
    return cv2.resize(p, (w, h), interpolation=cv2.INTER_NEAREST)

cap = cv2.VideoCapture(VIDEO_PATH)
fig, axes = plt.subplots(len(TEST_FRAMES), 1, figsize=(14, 7 * len(TEST_FRAMES)))
for ax, fi in zip(axes, TEST_FRAMES):
    cap.set(cv2.CAP_PROP_POS_FRAMES, fi); ok, frame = cap.read()
    if not ok:
        continue
    da = infer_drivable(frame)
    color = np.zeros_like(frame); color[da == 1] = (0, 180, 0)
    vis = cv2.addWeighted(frame, 1.0, color, 0.45, 0)
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); ax.set_title(f"frame {fi} - Phase 7 (DeepLabV3+aug)"); ax.axis("off")
cap.release(); plt.tight_layout(); plt.show()

In [ ]:
# 8) Render the hill segment + export a clean single-file ONNX
START, N_FRAMES = 14500, 900
OUT_PATH = os.path.join(ADAS, "phase7_hills_segmented.mp4")
cap = cv2.VideoCapture(VIDEO_PATH)
fps_src = cap.get(cv2.CAP_PROP_FPS) or 30
W, H = int(cap.get(3)), int(cap.get(4))
cap.set(cv2.CAP_PROP_POS_FRAMES, START)
writer = cv2.VideoWriter(OUT_PATH, cv2.VideoWriter_fourcc(*"mp4v"), fps_src, (W, H))
for _ in tqdm(range(N_FRAMES)):
    ok, frame = cap.read()
    if not ok:
        break
    da = infer_drivable(frame)
    color = np.zeros_like(frame); color[da == 1] = (0, 180, 0)
    writer.write(cv2.addWeighted(frame, 1.0, color, 0.45, 0))
cap.release(); writer.release()
print("saved:", OUT_PATH)

# dynamo=False -> legacy exporter -> ONE self-contained .onnx file (no .onnx.data split)
dummy = torch.randn(1, 3, 432, 768).to(DEVICE)
try:
    torch.onnx.export(model, dummy, os.path.join(ADAS, "drivable_idd_deeplab_768x432.onnx"),
                      input_names=["image"], output_names=["seg"], opset_version=12, dynamo=False)
except TypeError:  # older torch without the dynamo kwarg
    torch.onnx.export(model, dummy, os.path.join(ADAS, "drivable_idd_deeplab_768x432.onnx"),
                      input_names=["image"], output_names=["seg"], opset_version=12)
print("ONNX exported (single file) to Drive")

## Using the Phase 7 model in the app
1. Download `drivable_idd_deeplab_best.pth` from `MyDrive/adas/` into the repo root.
2. In `config.py` set:
   - `LEARNED_ARCH = "deeplabv3"`
   - `LEARNED_MODEL_PATH = "drivable_idd_deeplab_best.pth"`
3. Run `python main.py --video dashcam.mp4`. (You can keep the LRASPP weights too and flip `LEARNED_ARCH` to compare.)

## Reading the results
- Compare the six frames against Phase 6c. Expect the biggest gains on **5000 (haze)** and **17000 (night)** from the augmentation, and tighter edges on **12000/15000 (hills)** from the ASPP head.
- Val drivable-IoU should match or beat 0.92; augmentation can slightly lower the clean-val IoU while making the model far more robust on hard frames - judge by the six frames, not IoU alone.
- CPU speed at run time: ~2.6 fps @768x432, ~5.7 fps @512x288 (set `LEARNED_INPUT_W/H` in config.py).